# Exploratory Data Analysis - Chest X-Ray Pneumonia Dataset

This notebook explores the dataset structure, class distribution, and visualizes sample images.

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path
import pandas as pd

## 1. Dataset Size

In [ ]:
DATA_DIR = Path("../data/chest_xray")

splits = ["train", "val", "test"]
classes = ["NORMAL", "PNEUMONIA"]

counts = {}
for split in splits:
    counts[split] = {}
    for cls in classes:
        folder = DATA_DIR / split / cls
        n = len(list(folder.glob("*"))) if folder.exists() else 0
        counts[split][cls] = n

df_counts = pd.DataFrame(counts).T
df_counts["Total"] = df_counts.sum(axis=1)
print("Dataset size per split and class:")
print(df_counts)
print(f"\nTotal images: {df_counts['Total'].sum()}")

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, split in enumerate(splits):
    values = [counts[split][cls] for cls in classes]
    colors = ["#2ecc71", "#e74c3c"]
    axes[i].bar(classes, values, color=colors)
    axes[i].set_title(f"{split.capitalize()} Set")
    axes[i].set_ylabel("Number of Images")
    for j, v in enumerate(values):
        axes[i].text(j, v + 20, str(v), ha="center", fontweight="bold")

plt.suptitle("Class Distribution Across Splits", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Check Class Imbalance

In [ ]:
for split in splits:
    normal = counts[split]["NORMAL"]
    pneumonia = counts[split]["PNEUMONIA"]
    total = normal + pneumonia
    if total > 0:
        ratio = pneumonia / normal if normal > 0 else float("inf")
        print(f"{split:>5s}: Normal={normal:>5d} ({normal/total*100:.1f}%) | "
              f"Pneumonia={pneumonia:>5d} ({pneumonia/total*100:.1f}%) | "
              f"Ratio (P/N)={ratio:.2f}")

## 4. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))

for row, cls in enumerate(classes):
    folder = DATA_DIR / "train" / cls
    images = sorted(folder.glob("*"))[:5]
    for col, img_path in enumerate(images):
        img = Image.open(img_path).convert("RGB")
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls, fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Sample Images from Training Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Image Size Distribution

In [ ]:
widths, heights = [], []
sample_folder = DATA_DIR / "train"

for cls in classes:
    folder = sample_folder / cls
    for img_path in list(folder.glob("*"))[:200]:
        try:
            img = Image.open(img_path)
            w, h = img.size
            widths.append(w)
            heights.append(h)
        except Exception:
            continue

print(f"Width  - min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height - min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color="steelblue", edgecolor="black")
ax1.set_title("Image Width Distribution")
ax1.set_xlabel("Width (pixels)")
ax2.hist(heights, bins=30, color="coral", edgecolor="black")
ax2.set_title("Image Height Distribution")
ax2.set_xlabel("Height (pixels)")
plt.tight_layout()
plt.show()